# Ride-Hailing Method Comparison (Year 2023, Daily)

"
"This notebook compares day-to-day ridership movement across four NYC ride-hailing services for the **full year 2023** at a **daily** level:
"
"- FHV
- HVFHV
- Green Taxi
- Yellow Taxi

"
"Outputs are saved to `data/02-processed/TLC_ridehail/analysis_yearly_daily`.
"
"This notebook does **not** use hourly July outputs.


In [ ]:
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare, pearsonr, ttest_rel

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)


In [ ]:
INPUT_DIR = Path('data/01-interim/TLC_ridehail')
OUTPUT_DIR = Path('data/02-processed/TLC_ridehail/analysis_yearly_daily')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SERVICE_FILES = {
    'fhv': '2023_For_Hire_Vehicles_Trip_Data_cleaned.csv',
    'hvfhv': '2023_High_Volume_FHV_Trip_Data_cleaned.csv',
    'green_taxi': '2023_Green_Taxi_Trip_Data_cleaned.csv',
    'yellow_taxi': '2023_Yellow_Taxi_Trip_Data_cleaned.csv',
}

OUTPUT_DIR


In [ ]:
def find_col(df: pd.DataFrame, names: list[str]) -> str:
    m = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in m:
            return m[n.lower()]
    raise KeyError(f'Missing expected column from {names}')


def benjamini_hochberg(p_values: np.ndarray) -> np.ndarray:
    p_values = np.asarray(p_values, dtype=float)
    n = len(p_values)
    order = np.argsort(p_values)
    ranked = p_values[order]

    adjusted_ranked = np.empty(n, dtype=float)
    prev = 1.0
    for i in range(n - 1, -1, -1):
        rank = i + 1
        val = ranked[i] * n / rank
        prev = min(prev, val)
        adjusted_ranked[i] = prev

    adjusted = np.empty(n, dtype=float)
    adjusted[order] = np.clip(adjusted_ranked, 0, 1)
    return adjusted


In [ ]:
daily_parts = []

for service, filename in SERVICE_FILES.items():
    df = pd.read_csv(INPUT_DIR / filename)

    date_col = find_col(df, ['by_day_pickup_datetime', 'pickup_datetime', 'date'])
    trip_col = find_col(df, ['trip_count', 'ride_count', 'count'])

    temp = df[[date_col, trip_col]].copy()
    temp.columns = ['date', 'trip_count']

    temp['date'] = pd.to_datetime(temp['date'], errors='coerce')
    temp['trip_count'] = pd.to_numeric(
        temp['trip_count'].astype(str).str.replace(',', '', regex=False),
        errors='coerce',
    )

    temp = temp.dropna(subset=['date', 'trip_count'])
    temp['date'] = temp['date'].dt.normalize()

    daily_service = temp.groupby('date', as_index=False)['trip_count'].sum()
    daily_service = daily_service.rename(columns={'trip_count': service})

    daily_service.rename(columns={service: 'trip_count'}).to_csv(
        OUTPUT_DIR / f'2023_{service}_daily_trip_counts.csv', index=False
    )

    daily_parts.append(daily_service)

summary = pd.DataFrame({
    'service': list(SERVICE_FILES.keys()),
    'rows': [len(d) for d in daily_parts],
    'total_trips': [int(d[s].sum()) for d, s in zip(daily_parts, SERVICE_FILES.keys())],
})
summary


In [ ]:
daily = daily_parts[0]
for d in daily_parts[1:]:
    daily = daily.merge(d, on='date', how='outer')

daily = daily.sort_values('date').reset_index(drop=True)
daily.to_csv(OUTPUT_DIR / '2023_daily_totals_by_service.csv', index=False)

all_daily = (
    daily[['date'] + list(SERVICE_FILES.keys())]
    .set_index('date')
    .sum(axis=1)
    .rename('trip_count')
    .reset_index()
)
all_daily.to_csv(OUTPUT_DIR / '2023_all_ridehail_daily_trip_counts.csv', index=False)

daily.head()


## Statistical Comparison Setup

We compare services using daily log ratios:

$$ \log\left(rac{	ext{today trips}}{	ext{yesterday trips}}ight) $$

Then we run:
- Friedman repeated-measures test (overall differences across services)
- Pairwise paired t-tests (mean difference in daily log change)
- Pairwise Pearson correlations (co-movement)


In [ ]:
services = list(SERVICE_FILES.keys())
complete = daily.dropna(subset=services).copy()

ratios = np.log(complete[services] / complete[services].shift(1))
ratios = ratios.replace([np.inf, -np.inf], np.nan).dropna()

ratios_with_date = ratios.copy()
ratios_with_date.insert(0, 'date', complete['date'].iloc[1:].values)
ratios_with_date.head()


In [ ]:
friedman = friedmanchisquare(*(ratios[s].values for s in services))
overall_test = pd.DataFrame([{
    'test': 'friedman_repeated_measures',
    'n_days': len(ratios),
    'statistic': float(friedman.statistic),
    'p_value': float(friedman.pvalue),
}])
overall_test


In [ ]:
pair_rows = []
for a, b in combinations(services, 2):
    t_res = ttest_rel(ratios[a], ratios[b], nan_policy='omit')
    diff = ratios[a] - ratios[b]
    pair_rows.append({
        'service_a': a,
        'service_b': b,
        'n_days': int(diff.notna().sum()),
        'mean_diff_log_ratio': float(diff.mean()),
        't_statistic': float(t_res.statistic),
        'p_value': float(t_res.pvalue),
    })

pairwise_ttests = pd.DataFrame(pair_rows)
pairwise_ttests['p_value_fdr_bh'] = benjamini_hochberg(pairwise_ttests['p_value'].to_numpy())
pairwise_ttests.sort_values('p_value_fdr_bh')


In [ ]:
corr_rows = []
for a, b in combinations(services, 2):
    r, p = pearsonr(ratios[a], ratios[b])
    corr_rows.append({
        'service_a': a,
        'service_b': b,
        'n_days': len(ratios),
        'pearson_r': float(r),
        'p_value': float(p),
    })

pairwise_correlations = pd.DataFrame(corr_rows)
pairwise_correlations['p_value_fdr_bh'] = benjamini_hochberg(pairwise_correlations['p_value'].to_numpy())
pairwise_correlations.sort_values('pearson_r', ascending=False)


In [ ]:
ratio_summary = ratios.describe().T.reset_index().rename(columns={'index': 'service'})

daily.to_csv(OUTPUT_DIR / '2023_daily_totals_by_service.csv', index=False)
ratios_with_date.to_csv(OUTPUT_DIR / '2023_daily_log_ratios_by_service.csv', index=False)
overall_test.to_csv(OUTPUT_DIR / '2023_overall_test.csv', index=False)
pairwise_ttests.to_csv(OUTPUT_DIR / '2023_pairwise_ttests.csv', index=False)
pairwise_correlations.to_csv(OUTPUT_DIR / '2023_pairwise_correlations.csv', index=False)
ratio_summary.to_csv(OUTPUT_DIR / '2023_ratio_summary.csv', index=False)

print('Saved yearly daily analysis outputs to', OUTPUT_DIR)
ratio_summary
